# LeetCode 692 — Top K Frequent Words bằng Python

Link đề: https://leetcode.com/problems/top-k-frequent-words/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sorting, Heap (Priority Queue), Trie, Bucket Sort, Counting

Skeleton LeetCode:

```python
class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        ...
```

Ghi chú: bài chỉ dùng list, dict và chuỗi của Python; mọi đáp án trong notebook đã được kiểm chứng bằng một oracle độc lập (sắp xếp bằng hàm so sánh viết tay).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trả về `k` từ **xuất hiện nhiều nhất**, sắp theo **tần suất giảm dần**; nếu cùng tần suất thì xếp theo **thứ tự từ điển (lexicographical)** tăng dần.
2. Nắm các công cụ Python: `collections.Counter`, `sorted` với `key` là một **tuple**, mẹo dùng số âm để sắp giảm dần, so sánh chuỗi theo thứ tự từ điển, `heapq` (`heappush`, `heappop`) và cách định nghĩa thứ tự riêng bằng `__lt__`, danh sách "xô" (bucket), và `dict` lồng nhau làm Trie.
3. Đi từ **chọn lặp đi lặp lại phần tử tốt nhất `O(u * k)`** → **đếm rồi sắp xếp theo khóa `(-tần suất, từ)`** → **min-heap kích thước `k` (đúng yêu cầu follow-up `O(n log k)`)** → **bucket sort theo tần suất** → **Trie duyệt theo thứ tự chữ cái rồi sắp ổn định**.
4. Hiểu mấu chốt: bài có **hai tiêu chí sắp xếp** và chỉ cần gói chúng vào **một khóa** `(-count, word)`; khi dùng heap thì phải đảo thứ tự ở tiêu chí thứ hai để loại đúng phần tử "tệ nhất".
5. Tránh các lỗi kinh điển: dùng `Counter.most_common(k)` rồi tưởng đã đúng thứ tự khi hòa, `reverse=True` làm ngược cả tiêu chí từ điển, heap tuple `(count, word)` loại sai từ khi hòa, quên đảo kết quả khi lấy từ heap, và sắp xếp chỉ theo tần suất.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` và số nguyên `k`, trả về `k` chuỗi **xuất hiện nhiều nhất**.

Kết quả phải **sắp xếp theo tần suất từ cao xuống thấp**. Các từ **cùng tần suất** được sắp theo **thứ tự từ điển** (từ điển tăng dần: `"i"` đứng trước `"love"`).

**Ví dụ 1**

- Input: `words = ["i","love","leetcode","i","love","coding"]`, `k = 2`
- Output: `["i","love"]`
- Giải thích: `"i"` và `"love"` đều xuất hiện 2 lần, nhiều nhất. `"i"` đứng trước `"love"` vì thứ tự từ điển nhỏ hơn.

**Ví dụ 2**

- Input: `words = ["the","day","is","sunny","the","the","the","sunny","is","is"]`, `k = 4`
- Output: `["the","is","sunny","day"]`
- Giải thích: số lần xuất hiện lần lượt là 4, 3, 2, 1.

**Ràng buộc:**

- `1 <= words.length <= 500`
- `1 <= words[i].length <= 10`
- `words[i]` chỉ gồm chữ cái thường tiếng Anh.
- `k` nằm trong `[1, số từ khác nhau trong words]`.

**Follow-up:** có thể giải trong `O(n log k)` thời gian và `O(n)` bộ nhớ phụ không?

Chú ý: kết quả **đã xác định duy nhất** nhờ luật xếp khi hòa, nên ta có thể so sánh với đáp án cứng (không cần hàm kiểm tra hợp lệ).

In [1]:
from typing import List, Optional
from collections import Counter
from functools import cmp_to_key
import heapq
import random
import time

examples = [
    ((["i", "love", "leetcode", "i", "love", "coding"], 2), ["i", "love"]),
    ((["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 4), ["the", "is", "sunny", "day"]),
]
for (words, k), expected in examples:
    print(words, "k =", k, "-> kỳ vọng", expected)

['i', 'love', 'leetcode', 'i', 'love', 'coding'] k = 2 -> kỳ vọng ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] k = 4 -> kỳ vọng ['the', 'is', 'sunny', 'day']


## 3. Kiến thức nền

### 3.1. Đếm bằng `dict` và `collections.Counter`

Bài toán trước hết là **đếm tần suất**. Cách thủ công: dùng `dict`, mỗi lần gặp từ `w` thì `cnt[w] = cnt.get(w, 0) + 1`. `collections.Counter(words)` làm đúng việc đó trong một dòng và trả về một đối tượng giống dict `{từ: số_lần}`.

In [2]:
words = ["i", "love", "leetcode", "i", "love", "coding"]

cnt = {}
for w in words:
    cnt[w] = cnt.get(w, 0) + 1
print("đếm tay :", cnt)

c = Counter(words)
print("Counter :", c)
print("c['i'] =", c["i"], "| c['abc'] =", c["abc"], "(từ chưa có -> 0, không báo lỗi)")
print("số từ khác nhau:", len(c))

đếm tay : {'i': 2, 'love': 2, 'leetcode': 1, 'coding': 1}
Counter : Counter({'i': 2, 'love': 2, 'leetcode': 1, 'coding': 1})
c['i'] = 2 | c['abc'] = 0 (từ chưa có -> 0, không báo lỗi)
số từ khác nhau: 4


### 3.2. So sánh chuỗi theo thứ tự từ điển

Python so sánh chuỗi **từng ký tự từ trái sang phải** theo mã ký tự: `"i" < "love"` vì `'i' < 'l'`. Nếu một chuỗi là **tiền tố** của chuỗi kia thì chuỗi ngắn hơn **nhỏ hơn**: `"a" < "ab"`. Với chữ thường tiếng Anh, đây chính là thứ tự từ điển.

In [3]:
print("i" < "love")        # True
print("a" < "ab")          # True: tiền tố ngắn hơn đứng trước
print("abc" < "abd")       # True
print("b" < "ab")          # False: 'b' > 'a' ở ký tự đầu
print(sorted(["love", "i", "coding", "leetcode", "a", "ab"]))

True
True
True
False
['a', 'ab', 'coding', 'i', 'leetcode', 'love']


### 3.3. `sorted` với `key` là một tuple, và mẹo số âm

`sorted(iterable, key=f)` sắp xếp theo giá trị `f(x)`. Nếu `f` trả về **tuple**, Python so sánh **phần tử đầu trước**, hòa mới so phần tử thứ hai, v.v. Mặc định `sorted` sắp **tăng dần**.

Muốn "tần suất **giảm dần**, từ **tăng dần**" ta đặt khóa là `(-tần_suất, từ)`: đổi dấu tần suất để lớn nhất thành "nhỏ nhất", còn từ giữ nguyên để tăng dần.

In [4]:
pairs = [("love", 2), ("i", 2), ("coding", 1), ("leetcode", 1)]
print(sorted(pairs, key=lambda p: (-p[1], p[0])))   # tần suất giảm, hòa thì từ tăng

# so sánh: sai nếu dùng reverse=True với khóa (count, word)
print(sorted(pairs, key=lambda p: (p[1], p[0]), reverse=True))   # từ bị đảo ngược khi hòa!

[('i', 2), ('love', 2), ('coding', 1), ('leetcode', 1)]
[('love', 2), ('i', 2), ('leetcode', 1), ('coding', 1)]


Với `reverse=True`, **cả** hai tiêu chí bị đảo nên khi hòa tần suất, từ lại xếp **giảm dần** (`love` trước `i`). Đây là lỗi hay gặp (xem mục 13).

### 3.4. Hàng đợi ưu tiên `heapq` (min-heap)

`heapq` biến một `list` thành **min-heap**: phần tử **nhỏ nhất** luôn ở `heap[0]`. `heappush(h, x)` thêm `x`, `heappop(h)` lấy ra phần tử nhỏ nhất, cả hai `O(log m)` với `m` là số phần tử trong heap. Muốn giữ **top `k`** ta dùng heap kích thước `k`: mỗi khi vượt quá `k` thì `heappop` bỏ phần tử nhỏ nhất ("tệ nhất").

In [5]:
h = []
for x in [5, 1, 8, 3, 9, 2]:
    heapq.heappush(h, x)
    if len(h) > 3:
        heapq.heappop(h)          # bỏ số nhỏ nhất, chỉ giữ 3 số lớn nhất
    print(f"thêm {x}: heap = {sorted(h)}")
print("3 số lớn nhất:", sorted(h, reverse=True))

thêm 5: heap = [5]
thêm 1: heap = [1, 5]
thêm 8: heap = [1, 5, 8]
thêm 3: heap = [3, 5, 8]
thêm 9: heap = [5, 8, 9]
thêm 2: heap = [5, 8, 9]
3 số lớn nhất: [9, 8, 5]


### 3.5. Tự định nghĩa thứ tự: phương thức `__lt__`

`heapq` dùng toán tử `<` để so sánh. Với đối tượng của lớp tự định nghĩa, ta viết `__lt__(self, other)` để **quy định "nhỏ hơn" nghĩa là gì**. Trong bài này, phần tử "nhỏ nhất" trong heap phải là phần tử **tệ nhất** (tần suất thấp nhất, và khi hòa thì từ **lớn nhất** theo từ điển, vì từ đó sẽ xếp sau cùng).

In [6]:
class Item:
    def __init__(self, word, count):
        self.word, self.count = word, count
    def __lt__(self, other):          # "self nhỏ hơn other" = self TỆ HƠN other
        if self.count != other.count:
            return self.count < other.count
        return self.word > other.word  # hòa tần suất: từ lớn hơn là tệ hơn
    def __repr__(self):
        return f"({self.word},{self.count})"

a, b = Item("love", 2), Item("i", 2)
print("love tệ hơn i?", a < b)         # True: cùng 2 lần, 'love' > 'i'
print(sorted([Item("i", 2), Item("love", 2), Item("coding", 1)]))   # từ tệ nhất đến tốt nhất

love tệ hơn i? True
[(coding,1), (love,2), (i,2)]


### 3.6. Danh sách các "xô" (bucket) và Trie bằng dict lồng nhau

**Bucket:** tần suất tối đa không vượt quá `n` (số từ), nên ta dùng mảng `buckets[c]` = danh sách các từ xuất hiện đúng `c` lần. Duyệt `c` từ `n` giảm dần là có thứ tự tần suất giảm.

**Trie:** cây tiền tố; mỗi nút là một `dict` `{ký_tự: nút_con}`. Duyệt các con theo thứ tự chữ cái sẽ liệt kê các từ theo **thứ tự từ điển**. Ta sẽ dùng cách này ở Cách 5.

In [7]:
buckets = [[] for _ in range(6 + 1)]
for w, c in Counter(["a", "b", "a", "c", "c", "c"]).items():
    buckets[c].append(w)
print("buckets:", buckets)

root = {}
for w in ["to", "tea", "ted"]:
    node = root
    for ch in w:
        node = node.setdefault(ch, {})     # tạo nút con nếu chưa có
    node["#"] = node.get("#", 0) + 1       # '#' đánh dấu hết từ + đếm
print("trie:", root)

buckets: [[], ['b'], ['a'], ['c'], [], [], []]
trie: {'t': {'o': {'#': 1}, 'e': {'a': {'#': 1}, 'd': {'#': 1}}}}


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `["the","day","is","sunny","the","the","the","sunny","is","is"]`, `k = 4`.

**Bước 1 — đếm:** `the` 4 lần, `is` 3 lần, `sunny` 2 lần, `day` 1 lần.

**Bước 2 — xếp:** lớn đến nhỏ theo số lần: `the (4)`, `is (3)`, `sunny (2)`, `day (1)`. Lấy 4 từ đầu: `["the","is","sunny","day"]`.

Với ví dụ 1: `i` 2 lần, `love` 2 lần, `leetcode` 1, `coding` 1. Hai nhóm hòa nhau:

- nhóm 2 lần: `i`, `love` → theo từ điển `i < love`;
- nhóm 1 lần: `coding`, `leetcode` → `coding < leetcode`.

Danh sách đầy đủ: `i, love, coding, leetcode`; lấy `k = 2` được `["i","love"]`.

**Điều cốt lõi:** thứ tự cuối cùng là một thứ tự **toàn phần**: hai từ khác nhau luôn có một từ đứng trước (nếu hòa tần suất, từ điển quyết định; hai từ khác nhau không bao giờ bằng nhau về từ điển). Vì thế đáp án **duy nhất**.

In [8]:
words = ["i", "love", "leetcode", "i", "love", "coding"]
cnt = Counter(words)
print("đếm:", dict(cnt))
ordered = sorted(cnt, key=lambda w: (-cnt[w], w))
print("thứ tự đầy đủ:", ordered)
print("k = 2 ->", ordered[:2])

đếm: {'i': 2, 'love': 2, 'leetcode': 1, 'coding': 1}
thứ tự đầy đủ: ['i', 'love', 'coding', 'leetcode']
k = 2 -> ['i', 'love']


## 5. Cách 1: Brute Force (chọn phần tử tốt nhất, lặp `k` lần)

**Ý tưởng:** không sắp xếp toàn bộ. Đếm tần suất của từng từ **khác nhau** (bằng `words.count(w)`, chậm nhưng dễ hiểu), rồi lặp `k` lần: mỗi lần **quét toàn bộ** các từ còn lại để tìm từ "tốt nhất" (tần suất lớn nhất; hòa thì từ nhỏ nhất theo từ điển), thêm vào kết quả và loại khỏi danh sách.

Đây là *selection* (chọn dần) thay vì *sort*.

In [9]:
def brute_force(words, k):
    unique = []
    for w in words:                       # lấy các từ khác nhau (giữ thứ tự xuất hiện)
        if w not in unique:
            unique.append(w)
    freq = {w: words.count(w) for w in unique}    # đếm chậm: O(n) cho mỗi từ

    result = []
    for _ in range(k):
        best = None
        for w in unique:                  # quét tìm từ tốt nhất
            if best is None or freq[w] > freq[best] or (freq[w] == freq[best] and w < best):
                best = w
        result.append(best)
        unique.remove(best)
    return result

for (words, k), expected in examples:
    got = brute_force(words, k)
    print(words, k, "->", got)
    assert got == expected

['i', 'love', 'leetcode', 'i', 'love', 'coding'] 2 -> ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] 4 -> ['the', 'is', 'sunny', 'day']


**Độ phức tạp:** `u` là số từ khác nhau, `n` là tổng số từ. Đếm bằng `words.count`: `O(n * u)`; chọn `k` lần, mỗi lần quét `O(u)`: `O(k * u)`. Tổng **Time `O(n * u + k * u)`**, Space `O(u)`.

**Nút thắt:** hai chỗ chậm: (1) đếm lại từ đầu cho mỗi từ; (2) quét lại toàn bộ mỗi lần chọn. Cải tiến: **đếm một lượt** bằng dict/`Counter` (`O(n)`), rồi dùng sắp xếp hoặc cấu trúc dữ liệu thích hợp thay cho quét lặp.

## 6. Ý tưởng thuật toán / bước đột phá

Hai bước:

1. **Đếm một lượt** bằng `Counter` để có `cnt[w]`.
2. **Gói hai tiêu chí vào một khóa**: khóa của từ `w` là `(-cnt[w], w)`. Khóa nhỏ hơn nghĩa là từ **tốt hơn** (tần suất lớn hơn, hoặc cùng tần suất nhưng từ điển nhỏ hơn). Chỉ cần lấy `k` từ có khóa nhỏ nhất, theo thứ tự tăng của khóa.

Từ khóa này suy ra nhiều cách:

- **Sắp xếp toàn bộ** theo khóa rồi cắt `k` phần tử đầu: `O(u log u)`.
- **Heap kích thước `k`**: giữ `k` từ tốt nhất, loại dần từ "tệ nhất": `O(u log k)`.
- **Bucket theo tần suất** rồi chỉ cần sắp xếp trong từng xô theo từ điển.
- **Trie** cho thứ tự từ điển "miễn phí", rồi sắp ổn định theo tần suất.

In [10]:
words = ["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"]
cnt = Counter(words)
for w in cnt:
    print(f"{w:6s} key = {(-cnt[w], w)}")
print("sắp theo khóa:", sorted(cnt, key=lambda w: (-cnt[w], w)))

the    key = (-4, 'the')
day    key = (-1, 'day')
is     key = (-3, 'is')
sunny  key = (-2, 'sunny')
sắp theo khóa: ['the', 'is', 'sunny', 'day']


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 chọn lặp (đã làm); Cách 2 đếm + sắp xếp theo khóa; Cách 3 min-heap kích thước `k` (follow-up); Cách 4 bucket sort theo tần suất; Cách 5 Trie + sắp ổn định.

### 7.1. Cách 2: Đếm bằng `Counter` rồi `sorted` theo `(-count, word)`

**Giải thích từng đoạn:**

1. `cnt = Counter(words)`: `O(n)`.
2. `sorted(cnt, key=lambda w: (-cnt[w], w))`: sắp các **khóa** của `cnt` (các từ khác nhau) theo tuple. `u` từ nên tốn `O(u log u)`.
3. `[:k]`: lấy `k` từ đầu.

In [11]:
def count_sort(words, k):
    cnt = Counter(words)
    return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]

for (words, k), expected in examples:
    got = count_sort(words, k)
    print(words, k, "->", got)
    assert got == expected

['i', 'love', 'leetcode', 'i', 'love', 'coding'] 2 -> ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] 4 -> ['the', 'is', 'sunny', 'day']


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        cnt = Counter(words)
        return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]
```

**Độ phức tạp:** Time `O(n + u log u)` (`u <= n`, nên viết `O(n log n)`); Space `O(u)`.

### 7.2. Cách 3: Min-heap kích thước `k` (đáp ứng follow-up `O(n log k)`)

**Ý tưởng:** giữ trong heap `k` từ tốt nhất đã thấy. Phần tử **ở đỉnh heap** là từ **tệ nhất** trong `k` từ đó; khi có từ mới và heap đã đầy, ta thêm rồi bỏ đỉnh (tức bỏ từ tệ nhất).

**Giải thích từng đoạn:**

1. Lớp `Item(word, count)` với `__lt__`: "nhỏ hơn" = "**tệ hơn**": tần suất thấp hơn, hoặc cùng tần suất nhưng từ **lớn hơn** theo từ điển (vì từ lớn hơn xếp sau).
2. Duyệt từng `(w, c)` trong `Counter`: `heappush`, nếu `len > k` thì `heappop` (bỏ từ tệ nhất).
3. Cuối cùng heap có `k` từ. Mỗi lần `heappop` cho từ **tệ nhất còn lại** nên thu được thứ tự **từ tệ đến tốt**; **đảo ngược** để được từ tốt đến tệ.

Lý do `__lt__` đảo thứ tự từ điển: ta muốn bỏ đi từ xếp sau cùng khi hòa tần suất.

In [12]:
class WordItem:
    __slots__ = ("word", "count")
    def __init__(self, word, count):
        self.word, self.count = word, count
    def __lt__(self, other):                 # "nhỏ hơn" = tệ hơn
        if self.count != other.count:
            return self.count < other.count
        return self.word > other.word

def heap_k(words, k):
    cnt = Counter(words)
    heap = []
    for w, c in cnt.items():
        heapq.heappush(heap, WordItem(w, c))
        if len(heap) > k:
            heapq.heappop(heap)              # bỏ từ tệ nhất
    result = []
    while heap:
        result.append(heapq.heappop(heap).word)   # từ tệ nhất trước
    return result[::-1]                            # đảo: tốt nhất trước

for (words, k), expected in examples:
    got = heap_k(words, k)
    print(words, k, "->", got)
    assert got == expected

['i', 'love', 'leetcode', 'i', 'love', 'coding'] 2 -> ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] 4 -> ['the', 'is', 'sunny', 'day']


**Code nộp LeetCode (Cách 3):**

```python
class WordItem:
    __slots__ = ("word", "count")

    def __init__(self, word, count):
        self.word, self.count = word, count

    def __lt__(self, other):
        if self.count != other.count:
            return self.count < other.count
        return self.word > other.word


class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        cnt = Counter(words)
        heap = []
        for w, c in cnt.items():
            heapq.heappush(heap, WordItem(w, c))
            if len(heap) > k:
                heapq.heappop(heap)
        result = []
        while heap:
            result.append(heapq.heappop(heap).word)
        return result[::-1]
```

**Độ phức tạp:** đếm `O(n)`; mỗi từ khác nhau làm tối đa một `heappush` và một `heappop` trên heap kích thước `k + 1`: `O(u log k)`. Tổng **Time `O(n + u log k)`** (đáp ứng `O(n log k)`); Space `O(u)` cho `Counter` cộng `O(k)` cho heap.

### 7.3. Cách 4: Bucket sort theo tần suất

**Ý tưởng:** tần suất là số nguyên từ `1` đến `n`, nên ta **không cần so sánh tần suất**: đặt mỗi từ vào "xô" theo tần suất của nó. Rồi đi từ xô tần suất lớn nhất xuống nhỏ nhất; trong **mỗi xô** sắp các từ theo từ điển, và thêm vào kết quả cho đến khi đủ `k`.

**Giải thích từng đoạn:**

1. `buckets = [[] for _ in range(n + 1)]`, đặt `buckets[c].append(w)`.
2. `for c in range(n, 0, -1)`: xét xô từ lớn xuống nhỏ. `buckets[c].sort()` sắp từ điển.
3. Thêm từng từ vào `result`; khi `len(result) == k` thì dừng ngay.

In [13]:
def bucket_sort(words, k):
    cnt = Counter(words)
    n = len(words)
    buckets = [[] for _ in range(n + 1)]
    for w, c in cnt.items():
        buckets[c].append(w)
    result = []
    for c in range(n, 0, -1):             # tần suất lớn -> nhỏ
        if not buckets[c]:
            continue
        buckets[c].sort()                 # cùng tần suất: từ điển tăng
        for w in buckets[c]:
            result.append(w)
            if len(result) == k:
                return result
    return result

for (words, k), expected in examples:
    got = bucket_sort(words, k)
    print(words, k, "->", got)
    assert got == expected

['i', 'love', 'leetcode', 'i', 'love', 'coding'] 2 -> ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] 4 -> ['the', 'is', 'sunny', 'day']


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        cnt = Counter(words)
        n = len(words)
        buckets = [[] for _ in range(n + 1)]
        for w, c in cnt.items():
            buckets[c].append(w)
        result = []
        for c in range(n, 0, -1):
            buckets[c].sort()
            for w in buckets[c]:
                result.append(w)
                if len(result) == k:
                    return result
        return result
```

**Độ phức tạp:** đếm `O(n)`; dựng xô `O(n + u)`; sắp xếp trong từng xô: tổng `O(u log u)` tệ nhất (nếu mọi từ cùng một xô), nhưng thường nhỏ hơn. Space `O(n)`. Ưu điểm: không so sánh tần suất, dừng sớm khi đủ `k`.

### 7.4. Cách 5: Trie (thứ tự từ điển miễn phí) rồi sắp ổn định theo tần suất

**Ý tưởng:** xây **Trie** chứa mọi từ kèm số lần. Duyệt Trie theo **thứ tự chữ cái** (DFS, các con sắp tăng) liệt kê các từ **đã theo thứ tự từ điển**. Sau đó **sắp xếp ổn định** (stable sort) theo `-count`: vì `sort` của Python **ổn định** (hai phần tử bằng khóa giữ nguyên thứ tự ban đầu), các từ cùng tần suất vẫn giữ thứ tự từ điển.

**Giải thích từng đoạn:**

1. Chèn từng từ: đi theo ký tự, tạo nút con khi cần; cuối từ tăng `node["#"]`.
2. `dfs(node, prefix)`: nếu nút có `"#"` thì đây là một từ hoàn chỉnh `prefix` (từ ngắn hơn, là tiền tố, được thêm **trước** các từ dài hơn: đúng thứ tự từ điển); rồi duyệt các con theo ký tự tăng.
3. `out.sort(key=lambda x: -x[1])`: sắp ổn định theo tần suất giảm.

In [14]:
def trie_sort(words, k):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["#"] = node.get("#", 0) + 1

    out = []
    def dfs(node, prefix):
        if "#" in node:
            out.append((prefix, node["#"]))
        for ch in sorted(c for c in node if c != "#"):    # theo thứ tự chữ cái
            dfs(node[ch], prefix + ch)
    dfs(root, "")

    out.sort(key=lambda x: -x[1])         # sort ổn định: giữ thứ tự từ điển khi hòa
    return [w for w, _ in out[:k]]

for (words, k), expected in examples:
    got = trie_sort(words, k)
    print(words, k, "->", got)
    assert got == expected

['i', 'love', 'leetcode', 'i', 'love', 'coding'] 2 -> ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] 4 -> ['the', 'is', 'sunny', 'day']


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        root = {}
        for w in words:
            node = root
            for ch in w:
                node = node.setdefault(ch, {})
            node["#"] = node.get("#", 0) + 1

        out = []

        def dfs(node, prefix):
            if "#" in node:
                out.append((prefix, node["#"]))
            for ch in sorted(c for c in node if c != "#"):
                dfs(node[ch], prefix + ch)

        dfs(root, "")
        out.sort(key=lambda x: -x[1])
        return [w for w, _ in out[:k]]
```

**Độ phức tạp:** xây Trie `O(L)` với `L` là tổng độ dài các từ (tối đa `500 * 10`); duyệt `O(số nút Trie * 26)` tệ nhất; sắp ổn định `O(u log u)`. Space `O(L)`. Ở bài này `L <= 5000` nên rất nhẹ; Trie có lợi khi từ dài, nhiều tiền tố chung, hoặc cần thứ tự từ điển mà không so sánh chuỗi.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 (đếm + sắp theo khóa) với ví dụ 1

`words = ["i","love","leetcode","i","love","coding"]`, `k = 2`.

| Bước | Trạng thái |
|---|---|
| Đếm | `i:2, love:2, leetcode:1, coding:1` |
| Khóa `(-count, word)` | `i:(-2,'i')`, `love:(-2,'love')`, `leetcode:(-1,'leetcode')`, `coding:(-1,'coding')` |
| Sắp tăng theo khóa | `(-2,'i') < (-2,'love') < (-1,'coding') < (-1,'leetcode')` |
| Cắt `[:2]` | `['i','love']` |

In [15]:
def trace_sort(words, k):
    cnt = Counter(words)
    print("đếm:", dict(cnt))
    keys = {w: (-cnt[w], w) for w in cnt}
    for w, key in keys.items():
        print(f"  khóa của {w!r}: {key}")
    ordered = sorted(cnt, key=lambda w: keys[w])
    print("sau khi sắp:", ordered)
    print(f"lấy {k} từ đầu:", ordered[:k])
    return ordered[:k]

trace_sort(["i", "love", "leetcode", "i", "love", "coding"], 2)

đếm: {'i': 2, 'love': 2, 'leetcode': 1, 'coding': 1}
  khóa của 'i': (-2, 'i')
  khóa của 'love': (-2, 'love')
  khóa của 'leetcode': (-1, 'leetcode')
  khóa của 'coding': (-1, 'coding')
sau khi sắp: ['i', 'love', 'coding', 'leetcode']
lấy 2 từ đầu: ['i', 'love']


['i', 'love']

### 8.2. Trace Cách 3 (min-heap kích thước `k`) với ví dụ 2, `k = 2`

Đếm: `the:4, day:1, is:3, sunny:2`. Giả sử duyệt theo thứ tự xuất hiện: `the, day, is, sunny`. Ở mỗi bước ta thêm một từ; nếu heap có hơn `k = 2` phần tử, bỏ phần tử **tệ nhất** (đỉnh).

| Thêm | Heap sau khi thêm (từ,đếm) | `len > 2`? | Bỏ | Heap còn lại |
|---|---|---|---|---|
| `the:4` | `{the:4}` | không | | `{the:4}` |
| `day:1` | `{day:1, the:4}` | không | | `{day:1, the:4}` |
| `is:3` | `{day:1, the:4, is:3}` | có | `day:1` (tệ nhất) | `{is:3, the:4}` |
| `sunny:2` | `{sunny:2, the:4, is:3}` | có | `sunny:2` | `{is:3, the:4}` |

Cuối: pop theo thứ tự tệ → tốt: `is`, `the` → đảo → `["the","is"]`.

In [16]:
def trace_heap(words, k):
    cnt = Counter(words)
    print("đếm:", dict(cnt), "| k =", k)
    heap = []
    show = lambda h: sorted([(it.word, it.count) for it in h], key=lambda p: (p[1], p[0]))
    for w, c in cnt.items():
        heapq.heappush(heap, WordItem(w, c))
        msg = f"thêm {w}:{c} -> heap {show(heap)}"
        if len(heap) > k:
            worst = heapq.heappop(heap)
            msg += f" | quá {k} phần tử, bỏ tệ nhất {worst.word}:{worst.count} -> {show(heap)}"
        print(msg)
    popped = []
    while heap:
        it = heapq.heappop(heap)
        popped.append(it.word)
        print(f"pop {it.word}:{it.count}")
    print("pop theo thứ tự tệ -> tốt:", popped, "| đảo:", popped[::-1])
    return popped[::-1]

trace_heap(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 2)

đếm: {'the': 4, 'day': 1, 'is': 3, 'sunny': 2} | k = 2
thêm the:4 -> heap [('the', 4)]
thêm day:1 -> heap [('day', 1), ('the', 4)]
thêm is:3 -> heap [('day', 1), ('is', 3), ('the', 4)] | quá 2 phần tử, bỏ tệ nhất day:1 -> [('is', 3), ('the', 4)]
thêm sunny:2 -> heap [('sunny', 2), ('is', 3), ('the', 4)] | quá 2 phần tử, bỏ tệ nhất sunny:2 -> [('is', 3), ('the', 4)]
pop is:3
pop the:4
pop theo thứ tự tệ -> tốt: ['is', 'the'] | đảo: ['the', 'is']


['the', 'is']

Một ví dụ **có hòa tần suất** để thấy vì sao `__lt__` đảo từ điển: `words = ["b","a","c","a","b","c"]` (cả ba 2 lần), `k = 2`. Đáp án đúng `["a","b"]`; heap phải loại `c` (từ điển lớn nhất).

In [17]:
res = trace_heap(["b", "a", "c", "a", "b", "c"], 2)
assert res == ["a", "b"]

đếm: {'b': 2, 'a': 2, 'c': 2} | k = 2
thêm b:2 -> heap [('b', 2)]
thêm a:2 -> heap [('a', 2), ('b', 2)]
thêm c:2 -> heap [('a', 2), ('b', 2), ('c', 2)] | quá 2 phần tử, bỏ tệ nhất c:2 -> [('a', 2), ('b', 2)]
pop b:2
pop a:2
pop theo thứ tự tệ -> tốt: ['b', 'a'] | đảo: ['a', 'b']


### 8.3. Trace Cách 4 (bucket) với ví dụ 2, `k = 3`

Đếm: `the:4, day:1, is:3, sunny:2`, `n = 10`. Các xô khác rỗng: `buckets[4] = [the]`, `buckets[3] = [is]`, `buckets[2] = [sunny]`, `buckets[1] = [day]`. Đi từ `c = 10` xuống: các xô `10..5` rỗng; `c = 4` thêm `the`; `c = 3` thêm `is`; `c = 2` thêm `sunny` → đủ 3, dừng: `["the","is","sunny"]`.

In [18]:
def trace_bucket(words, k):
    cnt = Counter(words)
    n = len(words)
    buckets = [[] for _ in range(n + 1)]
    for w, c in cnt.items():
        buckets[c].append(w)
    print("xô khác rỗng:", {c: b for c, b in enumerate(buckets) if b})
    result = []
    for c in range(n, 0, -1):
        if not buckets[c]:
            continue
        buckets[c].sort()
        print(f"xô {c}: {buckets[c]} (đã sắp từ điển)")
        for w in buckets[c]:
            result.append(w)
            print(f"   thêm {w!r} -> {result}")
            if len(result) == k:
                print("   đủ k, dừng")
                return result
    return result

trace_bucket(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 3)

xô khác rỗng: {1: ['day'], 2: ['sunny'], 3: ['is'], 4: ['the']}
xô 4: ['the'] (đã sắp từ điển)
   thêm 'the' -> ['the']
xô 3: ['is'] (đã sắp từ điển)
   thêm 'is' -> ['the', 'is']
xô 2: ['sunny'] (đã sắp từ điển)
   thêm 'sunny' -> ['the', 'is', 'sunny']
   đủ k, dừng


['the', 'is', 'sunny']

## 9. Code nộp LeetCode

Ta nộp **Cách 2** (đếm + `sorted`) vì ngắn, dễ kiểm tra và đủ nhanh cho `n <= 500`. Nếu người phỏng vấn nhắc đến follow-up `O(n log k)`, dùng Cách 3 (min-heap kích thước `k`).

In [19]:
class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        cnt = Counter(words)                                # đếm tần suất
        return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]  # tần suất giảm, hòa thì từ điển tăng


sol = Solution()
for (words, k), expected in examples:
    got = sol.topKFrequent(words, k)
    print(words, k, "->", got, "| kỳ vọng", expected)
    assert got == expected

['i', 'love', 'leetcode', 'i', 'love', 'coding'] 2 -> ['i', 'love'] | kỳ vọng ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] 4 -> ['the', 'is', 'sunny', 'day'] | kỳ vọng ['the', 'is', 'sunny', 'day']


## 10. Vì sao đúng

**Thứ tự toàn phần.** Với hai từ khác nhau `a`, `b`, khóa `(-cnt[a], a)` và `(-cnt[b], b)` **không bao giờ bằng nhau** (vì `a != b`), nên thứ tự theo khóa là toàn phần và `sorted` cho kết quả **duy nhất**, chính là thứ tự mà đề yêu cầu: tần suất giảm dần (đổi dấu để thành tăng dần theo khóa), hòa thì từ điển tăng dần.

**Lấy `k` đầu.** `k` từ tốt nhất theo thứ tự toàn phần chính là `k` phần tử đầu của danh sách đã sắp.

**Min-heap kích thước `k` đúng.** Bất biến: sau khi xử lý một tập con các từ, heap chứa đúng `k` từ tốt nhất (theo thứ tự toàn phần) trong tập đó. Khi thêm từ mới `x`: nếu heap có `k + 1` phần tử, phần tử tệ nhất (đỉnh) không thể thuộc top `k` của tập mới (có `k` từ tốt hơn nó), nên bỏ nó đi là an toàn.

**Bucket đúng.** Duyệt xô tần suất giảm dần cho đúng thứ tự tiêu chí thứ nhất; sắp từ điển trong xô cho tiêu chí thứ hai; mỗi từ nằm đúng một xô.

**Trie + sort ổn định đúng.** DFS theo chữ cái liệt kê từ theo thứ tự từ điển; sắp ổn định theo `-count` chỉ hoán đổi thứ tự **giữa** các nhóm tần suất khác nhau, còn trong nhóm hòa giữ nguyên thứ tự từ điển.

Kiểm tra bằng code: **oracle độc lập** sắp xếp bằng một hàm so sánh `cmp` viết tay (không dùng khóa tuple), và mọi cách khác phải khớp.

In [20]:
def oracle(words, k):
    """Oracle: đếm bằng list.count, sắp xếp bằng hàm so sánh viết tay (không dùng khóa tuple)."""
    uniq = sorted(set(words))
    freq = {w: words.count(w) for w in uniq}
    def cmp(a, b):
        if freq[a] != freq[b]:
            return -1 if freq[a] > freq[b] else 1      # tần suất lớn hơn đứng trước
        if a != b:
            return -1 if a < b else 1                  # hòa: từ điển nhỏ hơn đứng trước
        return 0
    return sorted(uniq, key=cmp_to_key(cmp))[:k]

for (words, k), expected in examples:
    assert oracle(words, k) == expected
print("oracle khớp 2 ví dụ")

# tính chất: kết quả có đúng k từ khác nhau, tần suất không tăng, hòa thì từ điển tăng
rng = random.Random(692)
for _ in range(300):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 30))]
    u = len(set(words))
    k = rng.randint(1, u)
    res = Solution().topKFrequent(words, k)
    cnt = Counter(words)
    assert len(res) == k == len(set(res))
    for x, y in zip(res, res[1:]):
        assert cnt[x] > cnt[y] or (cnt[x] == cnt[y] and x < y)
    # mọi từ ngoài kết quả phải "tệ hơn" từ cuối của kết quả
    last = res[-1]
    for w in cnt:
        if w not in res:
            assert cnt[w] < cnt[last] or (cnt[w] == cnt[last] and w > last)
print("300 input ngẫu nhiên: kết quả đúng thứ tự và đúng tập top k")

oracle khớp 2 ví dụ
300 input ngẫu nhiên: kết quả đúng thứ tự và đúng tập top k


## 11. Time/Space Complexity

Gọi `n` là số từ trong `words`, `u` là số từ khác nhau (`u <= n`), `L` là tổng độ dài các từ.

| Cách | Time | Space |
|---|---|---|
| 1. Chọn lặp `k` lần | `O(n * u + k * u)` | `O(u)` |
| 2. Đếm + `sorted` | `O(n + u log u)` | `O(u)` |
| 3. Min-heap kích thước `k` | `O(n + u log k)` | `O(u)` (heap `O(k)`) |
| 4. Bucket sort | `O(n + u log u)` tệ nhất | `O(n)` |
| 5. Trie + sort ổn định | `O(L + u log u)` | `O(L)` |

Chú ý khi so sánh chuỗi: mỗi lần so sánh hai từ mất `O(độ dài từ)` (tối đa `10`), nên các cách dựa trên so sánh có thêm hệ số độ dài từ; với ràng buộc `<= 10` ta coi là hằng số.

Cách 3 đáp ứng đúng follow-up (`O(n log k)` thời gian, `O(n)` bộ nhớ phụ). Với `n <= 500` mọi cách đều rất nhanh.

## 12. Bộ test

Nhóm test: ví dụ đề, edge case (`k = 1`, `k = u`, toàn bộ từ giống nhau, toàn bộ khác nhau, hòa tần suất, từ là tiền tố của nhau), phản ví dụ cho các cách làm sai, đối chiếu **5 cách + lớp nộp** với **oracle** trên input ngẫu nhiên có seed (đúng ràng buộc), và test kích thước lớn nhất (`500` từ dài `10`).

In [21]:
methods = {
    "brute_force": brute_force,
    "count_sort": count_sort,
    "heap_k": heap_k,
    "bucket_sort": bucket_sort,
    "trie_sort": trie_sort,
    "Solution": Solution().topKFrequent,
}

# 12.1 ví dụ của đề
for (words, k), expected in examples:
    for name, fn in methods.items():
        assert fn(list(words), k) == expected, name
print("2 ví dụ: cả", len(methods), "cách đúng")

# 12.2 edge case
edge = [
    ((["a"], 1), ["a"]),                                      # một từ
    ((["a", "a", "a"], 1), ["a"]),                            # toàn bộ giống nhau
    ((["c", "b", "a"], 3), ["a", "b", "c"]),                  # toàn bộ khác nhau: thuần từ điển
    ((["c", "b", "a"], 1), ["a"]),                            # k = 1, hòa 3 chiều
    ((["b", "a", "c", "a", "b", "c"], 2), ["a", "b"]),        # hòa: phải loại c
    ((["a", "ab", "a", "ab", "abc"], 3), ["a", "ab", "abc"]), # tiền tố: 'a' < 'ab' < 'abc'
    ((["ab", "a", "ab", "a"], 2), ["a", "ab"]),               # tiền tố, hòa tần suất
    ((["x", "y", "y", "z", "z", "z"], 3), ["z", "y", "x"]),   # tần suất 3, 2, 1: không hòa
    ((["i", "love", "leetcode", "i", "love", "coding"], 4), ["i", "love", "coding", "leetcode"]),  # k = u
]
for (words, k), expected in edge:
    assert oracle(list(words), k) == expected, (words, k)
    for name, fn in methods.items():
        assert fn(list(words), k) == expected, (name, words, k, fn(list(words), k))
print("edge case: OK")

2 ví dụ: cả 6 cách đúng
edge case: OK


In [22]:
# 12.3 phản ví dụ: các cách làm SAI phổ biến
def wrong_most_common(words, k):                 # dùng Counter.most_common, hòa thì theo thứ tự xuất hiện
    return [w for w, _ in Counter(words).most_common(k)]

def wrong_reverse(words, k):                     # reverse=True làm ngược cả từ điển
    cnt = Counter(words)
    return sorted(cnt, key=lambda w: (cnt[w], w), reverse=True)[:k]

def wrong_heap_tuple(words, k):                  # heap tuple (count, word): loại sai từ khi hòa
    cnt = Counter(words)
    heap = []
    for w, c in cnt.items():
        heapq.heappush(heap, (c, w))
        if len(heap) > k:
            heapq.heappop(heap)
    return [w for c, w in sorted(heap, key=lambda p: (-p[0], p[1]))]

def wrong_no_reverse(words, k):                  # quên đảo kết quả lấy từ heap
    cnt = Counter(words)
    heap = []
    for w, c in cnt.items():
        heapq.heappush(heap, WordItem(w, c))
        if len(heap) > k:
            heapq.heappop(heap)
    return [heapq.heappop(heap).word for _ in range(len(heap))]

def wrong_only_count(words, k):                  # chỉ sắp theo tần suất (sort ổn định giữ thứ tự xuất hiện)
    cnt = Counter(words)
    return sorted(cnt, key=lambda w: -cnt[w])[:k]

w1 = ["b", "a", "c", "a", "b", "c"]
print("đúng               :", Solution().topKFrequent(w1, 2))
print("most_common        :", wrong_most_common(w1, 2))
print("reverse=True       :", wrong_reverse(w1, 2))
print("heap tuple (c, w)  :", wrong_heap_tuple(w1, 2))
print("quên đảo từ heap   :", wrong_no_reverse(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 3))
print("chỉ sắp theo count :", wrong_only_count(w1, 2))
assert wrong_most_common(w1, 2) != ["a", "b"]
assert wrong_reverse(w1, 2) != ["a", "b"]
assert wrong_heap_tuple(w1, 2) != ["a", "b"]
assert wrong_no_reverse(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 3) != ["the", "is", "sunny"]
assert wrong_only_count(w1, 2) != ["a", "b"]
print("các cách sai bị phát hiện")

đúng               : ['a', 'b']
most_common        : ['b', 'a']
reverse=True       : ['c', 'b']
heap tuple (c, w)  : ['b', 'c']
quên đảo từ heap   : ['sunny', 'is', 'the']
chỉ sắp theo count : ['b', 'a']
các cách sai bị phát hiện


In [23]:
# 12.4 đối chiếu mọi cách với oracle trên input ngẫu nhiên (seed cố định); bảng chữ cái nhỏ để tạo nhiều trường hợp hòa
rng = random.Random(20240692)
cases = 0
for _ in range(1500):
    alpha = rng.choice(["ab", "abc", "abcd", "abcdefgh"])
    max_len = rng.choice([1, 2, 3])
    n = rng.randint(1, 40)
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, max_len))) for _ in range(n)]
    k = rng.randint(1, len(set(words)))
    expected = oracle(words, k)
    for name, fn in methods.items():
        assert fn(list(words), k) == expected, (name, words, k)
    cases += 1
print("đối chiếu", cases, "input ngẫu nhiên với", len(methods), "cách: OK")

đối chiếu 1500 input ngẫu nhiên với 6 cách: OK


In [24]:
# 12.5 test lớn đúng ràng buộc: 500 từ, độ dài tới 10, nhiều từ khác nhau
rng = random.Random(500)
big_cases = {
    "từ ngẫu nhiên dài 10": ["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10)) for _ in range(500)],
    "ít từ, nhiều lần": [rng.choice(["alpha", "beta", "gamma", "delta", "eps"]) for _ in range(500)],
    "toàn bộ khác nhau": [f"w{i:03d}" for i in range(500)][::-1],
    "một từ lặp 500 lần": ["zzzzzzzzzz"] * 500,
    "bảng chữ cái 2": ["".join(rng.choice("ab") for _ in range(rng.randint(1, 10))) for _ in range(500)],
}
for title, words in big_cases.items():
    u = len(set(words))
    for k in {1, max(1, u // 2), u}:
        expected = oracle(words, k)
        t0 = time.perf_counter()
        for name, fn in methods.items():
            assert fn(list(words), k) == expected, (title, name, k)
        dt = time.perf_counter() - t0
    print(f"{title:22s} u = {u:3d}: mọi cách khớp oracle")
print("test lớn: OK")

từ ngẫu nhiên dài 10   u = 500: mọi cách khớp oracle
ít từ, nhiều lần       u =   5: mọi cách khớp oracle


toàn bộ khác nhau      u = 500: mọi cách khớp oracle
một từ lặp 500 lần     u =   1: mọi cách khớp oracle
bảng chữ cái 2         u = 272: mọi cách khớp oracle
test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `Counter.most_common(k)` và tưởng đã xong

`most_common` sắp theo tần suất giảm dần nhưng khi **hòa**, thứ tự là **thứ tự các từ xuất hiện lần đầu**, **không** phải thứ tự từ điển.

In [25]:
words = ["b", "a", "c", "a", "b", "c"]
print("most_common:", [w for w, _ in Counter(words).most_common(2)], "(sai)")
print("ĐÚNG      :", Solution().topKFrequent(words, 2))

most_common: ['b', 'a'] (sai)
ĐÚNG      : ['a', 'b']


**Vì sao sai:** cả `b`, `a`, `c` xuất hiện 2 lần, nên `most_common` trả theo thứ tự xuất hiện đầu tiên: `b, a`. Đề yêu cầu từ điển: `a, b`. **Sửa:** dùng `sorted` với khóa `(-count, word)`.

### Lỗi 2: `reverse=True` với khóa `(count, word)`

`reverse=True` đảo **tất cả** tiêu chí, nên từ điển khi hòa bị sắp giảm dần.

In [26]:
words = ["i", "love", "leetcode", "i", "love", "coding"]
cnt = Counter(words)
print("SAI  :", sorted(cnt, key=lambda w: (cnt[w], w), reverse=True)[:2])
print("ĐÚNG :", sorted(cnt, key=lambda w: (-cnt[w], w))[:2])

SAI  : ['love', 'i']
ĐÚNG : ['i', 'love']


**Vì sao sai:** với `reverse=True`, `love` (lớn hơn `i`) được xếp trước. **Sửa:** không dùng `reverse`; đổi dấu **riêng** tần suất: `(-cnt[w], w)`.

### Lỗi 3: Heap tuple `(count, word)` loại sai từ khi hòa

Min-heap tuple `(count, word)` bỏ phần tử **nhỏ nhất** tức tần suất thấp nhất, và khi hòa thì **từ nhỏ nhất theo từ điển**, nhưng ta cần bỏ từ **lớn nhất** (xếp sau cùng).

In [27]:
words = ["b", "a", "c", "a", "b", "c"]
print("SAI  :", wrong_heap_tuple(words, 2))
print("ĐÚNG :", heap_k(words, 2))

SAI  : ['b', 'c']
ĐÚNG : ['a', 'b']


**Vì sao sai:** khi heap có `(2,'a'), (2,'b'), (2,'c')`, đỉnh là `(2,'a')`: từ **tốt nhất** bị loại. **Sửa:** dùng lớp có `__lt__` đảo thứ tự từ điển (hoặc mã hóa từ thành khóa đảo, ví dụ dùng `(count, tuple(-ord(c) ...))` thận trọng).

### Lỗi 4: Quên đảo kết quả từ heap

Heap trả về từ **tệ nhất trước**. Nếu `append` từng lần `heappop` mà không đảo, kết quả ngược thứ tự.

In [28]:
words = ["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"]
print("SAI  :", wrong_no_reverse(words, 3))
print("ĐÚNG :", heap_k(words, 3))

SAI  : ['sunny', 'is', 'the']
ĐÚNG : ['the', 'is', 'sunny']


**Vì sao sai:** `heappop` lần đầu cho `sunny:2` (tệ nhất trong top 3). **Sửa:** `result[::-1]` hoặc lấp đầy mảng từ cuối.

### Lỗi 5: Chỉ sắp theo tần suất

Nếu bỏ tiêu chí từ điển, `sorted` (ổn định) giữ thứ tự xuất hiện của `Counter` khi hòa, nên kết quả đúng "tình cờ" ở một số input và sai ở input khác.

In [29]:
print("SAI  :", wrong_only_count(["b", "a", "c", "a", "b", "c"], 2))
print("ĐÚNG :", Solution().topKFrequent(["b", "a", "c", "a", "b", "c"], 2))
print("tình cờ đúng:", wrong_only_count(["a", "b", "c", "a", "b", "c"], 2))

SAI  : ['b', 'a']
ĐÚNG : ['a', 'b']
tình cờ đúng: ['a', 'b']


**Vì sao sai:** kết quả phụ thuộc vào thứ tự các từ xuất hiện trong input, mà đề thì không cho phép. **Sửa:** luôn dùng khóa đủ hai tiêu chí.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra:**

- "Top `k`", "`k` phần tử xuất hiện nhiều nhất/ít nhất", "sắp theo tần suất": **đếm bằng `Counter`**, rồi **sắp xếp** hoặc **heap**.
- Có **hai tiêu chí** (tần suất rồi từ điển): gói thành **một khóa tuple**.
- Follow-up `O(n log k)`: **heap kích thước `k`**.
- Tần suất bị chặn bởi `n`: **bucket sort** cho `O(n)` (không so sánh).

**Mẹo:**

1. **Sắp giảm dần theo số, tăng dần theo chuỗi:** đổi dấu số (`-count`), đừng dùng `reverse=True` cho cả khóa.
2. Heap `heapq` luôn là **min-heap**: muốn bỏ phần tử "tệ nhất" thì định nghĩa thứ tự sao cho "tệ nhất" là **nhỏ nhất** (dùng `__lt__` hoặc đổi dấu).
3. Với số: dùng `(count, -x)` cho heap; với chuỗi không đổi dấu được nên dùng lớp có `__lt__`.
4. `sorted` của Python **ổn định**: có thể sắp theo tiêu chí phụ trước, tiêu chí chính sau (hai lượt sắp).
5. `heapq.nsmallest(k, cnt, key=lambda w: (-cnt[w], w))` là cách ngắn gọn tương đương Cách 3 (thư viện tự dùng heap).
6. Luôn tự tạo input **hòa tần suất** để kiểm tra tiêu chí thứ hai.

## 15. Biến thể

### 15.1. `heapq.nsmallest` / `nlargest` với khóa

Thư viện có sẵn hàm chọn `k` phần tử nhỏ nhất theo một khóa, dùng heap bên trong, và trả về kết quả **đã sắp xếp**:

In [30]:
def nsmallest_version(words, k):
    cnt = Counter(words)
    return heapq.nsmallest(k, cnt, key=lambda w: (-cnt[w], w))

for (words, k), expected in examples:
    assert nsmallest_version(words, k) == expected
rng = random.Random(151)
for _ in range(200):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 2))) for _ in range(rng.randint(1, 25))]
    k = rng.randint(1, len(set(words)))
    assert nsmallest_version(words, k) == oracle(words, k)
print("nsmallest với khóa (-count, word): khớp oracle")

nsmallest với khóa (-count, word): khớp oracle


### 15.2. Tần suất ít nhất (bottom `k`)

Đảo tiêu chí tần suất: `k` từ **ít xuất hiện nhất**, hòa thì từ điển tăng. Khóa trở thành `(cnt[w], w)`.

In [31]:
def bottom_k(words, k):
    cnt = Counter(words)
    return sorted(cnt, key=lambda w: (cnt[w], w))[:k]

print(bottom_k(["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"], 2))   # ['day', 'sunny']

['day', 'sunny']


### 15.3. Trả kèm tần suất, hoặc dữ liệu dạng luồng

- Trả về cặp `(từ, tần suất)`: giữ nguyên khóa, thay `[:k]` của danh sách từ bằng danh sách cặp.
- **Luồng dữ liệu** (từ đến liên tục): giữ `Counter` và cập nhật; hỏi top `k` bằng heap `O(u log k)` mỗi lần hỏi, hoặc duy trì cấu trúc sắp xếp (cây cân bằng) nếu hỏi rất nhiều.

### 15.4. Các bài cùng họ

- **LeetCode 347 — Top K Frequent Elements:** số nguyên, không có luật hòa; bucket sort `O(n)`.
- **LeetCode 451 — Sort Characters By Frequency:** sắp ký tự theo tần suất.
- **LeetCode 1636 — Sort Array by Increasing Frequency:** tần suất tăng, hòa thì giá trị giảm.
- **LeetCode 973 — K Closest Points to Origin:** top `k` theo khoảng cách bằng heap.
- **LeetCode 215 — Kth Largest Element in an Array:** quickselect hoặc heap kích thước `k`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chọn lặp `k` lần | quét tìm từ tốt nhất, lặp `k` lần | `O(n * u + k * u)` | `O(u)` | chỉ để hiểu đề, input rất nhỏ |
| 2. Đếm + `sorted` | khóa `(-count, word)` | `O(n + u log u)` | `O(u)` | **cách nộp chuẩn**, ngắn và rõ |
| 3. Min-heap kích thước `k` | giữ `k` tốt nhất, bỏ đỉnh (tệ nhất) | `O(n + u log k)` | `O(u)` | follow-up, `k` nhỏ hơn nhiều so với `u` |
| 4. Bucket sort | xô theo tần suất, sắp từ điển trong xô | `O(n + u log u)` tệ nhất | `O(n)` | khi tần suất bị chặn, muốn dừng sớm |
| 5. Trie + sort ổn định | duyệt Trie ra thứ tự từ điển, sắp ổn định theo tần suất | `O(L + u log u)` | `O(L)` | nhiều tiền tố chung, tránh so sánh chuỗi |

Kết luận: để thi, dùng Cách 2; nếu bị hỏi follow-up thì Cách 3. Cách 4 và 5 là các góc nhìn khác để hiểu thêm cấu trúc dữ liệu.

## 17. Bài tập vận dụng

Làm từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **chưa được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, đoán in ra gì:

```python
words = ["b", "b", "a", "c", "c", "d"]
cnt = Counter(words)
print(sorted(cnt, key=lambda w: (-cnt[w], w)))
print(sorted(cnt, key=lambda w: (-cnt[w], w))[:3])
```

### Bài 2 (tính tay)

Với `words = ["to","be","or","not","to","be","to","a"]` và `k = 3`: viết bảng đếm, khóa `(-count, word)` của mọi từ, thứ tự sắp xếp và kết quả. (Chú ý nhóm các từ hòa nhau ở tần suất 1.)

### Bài 3 (điền TODO)

Điền chỗ trống của Cách 4 (bucket sort):

In [32]:
def ex3_bucket(words, k):
    cnt = Counter(words)
    n = len(words)
    buckets = [[] for _ in range(...)]            # TODO 1: cần bao nhiêu xô? (tần suất tối đa là gì?)
    for w, c in cnt.items():
        buckets[...].append(w)                    # TODO 2: từ w vào xô nào?
    result = []
    for c in range(n, 0, -1):
        ...                                       # TODO 3: sắp từ điển các từ trong xô c
        for w in buckets[c]:
            result.append(w)
            if len(result) == k:
                return result
    return result

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert ex3_bucket(["i", "love", "leetcode", "i", "love", "coding"], 2) == ["i", "love"]

### Bài 4 (tự viết lại)

Viết lại **Cách 3 (min-heap kích thước `k`)** từ đầu, không nhìn bài giảng: lớp `WordItem` với `__lt__` (tệ hơn = nhỏ hơn), đẩy vào heap, bỏ đỉnh khi vượt `k`, rồi đảo kết quả.

In [33]:
def ex4_heap(words, k):
    # TODO: tự viết min-heap kích thước k với lớp có __lt__ (nhớ đảo từ điển khi hòa)
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex4_heap(["b", "a", "c", "a", "b", "c"], 2) == ["a", "b"]

### Bài 5 (mini challenge, biến thể)

Viết hàm `top_k_first_seen(words, k)`: giống đề nhưng khi hòa tần suất thì xếp theo **thứ tự xuất hiện đầu tiên** trong `words` (không phải từ điển). Gợi ý: ghi lại vị trí xuất hiện đầu tiên của mỗi từ và đưa nó vào khóa. Ví dụ `["b","a","c","a","b","c"]`, `k = 2` → `["b","a"]`.

In [34]:
def ex5_first_seen(words, k):
    # TODO: tần suất giảm dần; hòa thì theo vị trí xuất hiện đầu tiên (nhỏ trước)
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex5_first_seen(["b", "a", "c", "a", "b", "c"], 2) == ["b", "a"]

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): một từ có thể xuất hiện tối đa bao nhiêu lần? Từ đó suy ra chỉ số lớn nhất của xô.
- Mức 2 (vừa): xô thứ `c` chứa các từ có tần suất đúng bằng `c`.
- Mức 3 (gần đáp án): `range(n + 1)`; `buckets[c]`; `buckets[c].sort()`.

**Bài 4**

- Mức 1: heap phải loại từ "tệ nhất". Ai tệ nhất khi hòa tần suất?
- Mức 2: `__lt__`: nếu `count` khác nhau trả `self.count < other.count`, ngược lại trả `self.word > other.word`.
- Mức 3: `push` rồi `if len(heap) > k: pop`; cuối cùng pop hết và `[::-1]`.

**Bài 5**

- Mức 1: `{w: i for i, w in reversed(list(enumerate(words)))}` hoặc dùng `dict` giữ thứ tự chèn của `Counter`.
- Mức 2: `Counter` giữ thứ tự lần đầu gặp; có thể dùng `list(cnt)` để lấy thứ tự đó.
- Mức 3: `first = {w: i for i, w in enumerate(cnt)}` rồi `sorted(cnt, key=lambda w: (-cnt[w], first[w]))[:k]`.

## 19. Đáp án tham khảo

**Bài 1:** đếm: `b:2, a:1, c:2, d:1`. Khóa: `b:(-2,'b')`, `a:(-1,'a')`, `c:(-2,'c')`, `d:(-1,'d')`. Sắp tăng theo khóa: `(-2,'b') < (-2,'c') < (-1,'a') < (-1,'d')`. In ra `['b', 'c', 'a', 'd']` rồi `['b', 'c', 'a']`.

**Bài 2:** đếm: `to:3, be:2, or:1, not:1, a:1`. Khóa: `to:(-3,'to')`, `be:(-2,'be')`, `a:(-1,'a')`, `not:(-1,'not')`, `or:(-1,'or')`. Sắp xếp: `to, be, a, not, or`. Với `k = 3`: `["to","be","a"]`. (Không có hai từ hòa tần suất 2; các từ hòa là nhóm tần suất 1 gồm `a, not, or`, và `a` đứng đầu theo từ điển.)

In [35]:
# Bài 1
words = ["b", "b", "a", "c", "c", "d"]
cnt = Counter(words)
r1 = sorted(cnt, key=lambda w: (-cnt[w], w))
assert r1 == ["b", "c", "a", "d"] and r1[:3] == ["b", "c", "a"]
print("Bài 1:", r1, r1[:3])

# Bài 2
words = ["to", "be", "or", "not", "to", "be", "to", "a"]
assert dict(Counter(words)) == {"to": 3, "be": 2, "or": 1, "not": 1, "a": 1}
assert Solution().topKFrequent(words, 3) == ["to", "be", "a"] == oracle(words, 3)
assert Solution().topKFrequent(words, 5) == ["to", "be", "a", "not", "or"]
print("Bài 2:", Solution().topKFrequent(words, 3))

# Bài 3
def ex3_answer(words, k):
    cnt = Counter(words)
    n = len(words)
    buckets = [[] for _ in range(n + 1)]
    for w, c in cnt.items():
        buckets[c].append(w)
    result = []
    for c in range(n, 0, -1):
        buckets[c].sort()
        for w in buckets[c]:
            result.append(w)
            if len(result) == k:
                return result
    return result

# Bài 4
def ex4_answer(words, k):
    class Item:
        def __init__(self, word, count):
            self.word, self.count = word, count
        def __lt__(self, other):
            if self.count != other.count:
                return self.count < other.count
            return self.word > other.word
    cnt = Counter(words)
    heap = []
    for w, c in cnt.items():
        heapq.heappush(heap, Item(w, c))
        if len(heap) > k:
            heapq.heappop(heap)
    out = []
    while heap:
        out.append(heapq.heappop(heap).word)
    return out[::-1]

# Bài 5
def ex5_answer(words, k):
    cnt = Counter(words)
    first = {w: i for i, w in enumerate(cnt)}     # Counter giữ thứ tự lần đầu xuất hiện
    return sorted(cnt, key=lambda w: (-cnt[w], first[w]))[:k]

def ex5_oracle(words, k):
    uniq = []
    for w in words:
        if w not in uniq:
            uniq.append(w)
    best = sorted(uniq, key=lambda w: (-words.count(w), words.index(w)))
    return best[:k]

assert ex5_answer(["b", "a", "c", "a", "b", "c"], 2) == ["b", "a"]
rng = random.Random(19)
for _ in range(500):
    words = ["".join(rng.choice("abcd") for _ in range(rng.randint(1, 2))) for _ in range(rng.randint(1, 30))]
    k = rng.randint(1, len(set(words)))
    assert ex3_answer(words, k) == ex4_answer(words, k) == oracle(words, k)
    assert ex5_answer(words, k) == ex5_oracle(words, k)
print("Bài 3, 4, 5: khớp oracle trên 500 input ngẫu nhiên")

Bài 1: ['b', 'c', 'a', 'd'] ['b', 'c', 'a']
Bài 2: ['to', 'be', 'a']


Bài 3, 4, 5: khớp oracle trên 500 input ngẫu nhiên


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "top `k` theo tần suất, hòa theo từ điển":**

```python
class Solution:
    def topKFrequent(self, words: List[str], k: int) -> List[str]:
        cnt = Counter(words)
        return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]
```

**Template heap `O(n log k)` (follow-up):**

```python
class WordItem:
    __slots__ = ("word", "count")
    def __init__(self, word, count):
        self.word, self.count = word, count
    def __lt__(self, other):                  # nhỏ hơn = tệ hơn
        if self.count != other.count:
            return self.count < other.count
        return self.word > other.word         # hòa: từ lớn hơn tệ hơn

# heap = []; push từng WordItem; nếu len(heap) > k: heappop; cuối cùng pop hết rồi đảo
```

**Checklist trước khi submit:**

- [ ] Đã đếm tần suất (một lượt `Counter`)?
- [ ] Khóa sắp xếp là `(-count, word)`: **không** dùng `reverse=True` cho cả hai?
- [ ] Nếu dùng heap: thứ tự `__lt__` đảo từ điển khi hòa, và đã **đảo kết quả** khi lấy từ heap?
- [ ] Không dựa vào `most_common` cho thứ tự hòa?
- [ ] Đã thử input hòa tần suất, từ là tiền tố của nhau, `k = 1`, `k = số từ khác nhau`?
- [ ] Time `O(n log n)` (hoặc `O(n log k)` với heap), Space `O(n)`?

**Ghi nhớ một câu:** *"Hai tiêu chí thì gói vào một khóa (-đếm, từ); heap thì loại từ tệ nhất, khi hòa là từ lớn nhất theo từ điển."*